In [14]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

In [15]:
class TextDataset(Dataset):
    def __init__(self, text_file, seq_length=128):
        with open("alice.txt", 'r', encoding="utf-8") as f:
            text = f.read()
        words = text.split()

        unique_words = sorted(set(words))
        self.word_to_idx = {word: i for i, word in enumerate(unique_words)}
        self.idx_to_word = {i: word for i, word in enumerate(unique_words)}
        self.vocab_size = len(unique_words)

        self.data = [self.word_to_idx[word] for word in words]
        self.seq_length = seq_length
    def __len__(self):
        return len(self.data) - self.seq_length
    def __getitem__(self, idx):
        x = torch.tensor(self.data[idx:idx+self.seq_length])
        y = torch.tensor(self.data[idx+1:idx+self.seq_length+1])
        return x, y

In [16]:
class TransformerLM(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=4, dim_feedforward=512, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = nn.Embedding(5000, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        self.fc = nn.Linear(d_model, vocab_size)
        self.d_model = d_model
    def forward(self, x):
        seq_len = x.size(1)
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        x = self.embedding(x) + self.pos_encoder(positions)
        x = x.transpose(0, 1) 
        x = self.transformer(x)
        x = x.transpose(0, 1)
        return self.fc(x) # Transformer expects (seq_len, batch_size, d_model)

In [17]:
from tqdm import tqdm

dataset = TextDataset('alice.txt')
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

model = TransformerLM(vocab_size=dataset.vocab_size)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()  

for epoch in range(10):
    progress_bar = tqdm(dataloader, desc=f'Epoch {epoch}')
    for epoch in range(10):
        for x, y in dataloader:
            optimizer.zero_grad()
            output = model(x)
            loss = criterion(output.view(-1, dataset.vocab_size), y.view(-1))
            loss.backward()
            optimizer.step()

            progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
        print(f'Epoch {epoch}, Loss: {loss.item():.4f}')

C:\Users\DSU\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
Epoch 0:   0%|          | 0/920 [00:12<?, ?it/s, loss=7.0534]

KeyboardInterrupt: 